<a href="https://colab.research.google.com/github/teju917/GENAITUTORIALS050/blob/main/p6_PDF_RAG_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📄 PDF-Based RAG System (Chat with your PDF)
**Upload any PDF → ask questions → get answers ONLY from that PDF.**  
Questions that are not related to the PDF are politely refused.

✅ No API key ✅ Local open-source models only ✅ Works in Google Colab (GPU recommended, CPU also works)

## How it works
```
PDF upload ─► extract text (page by page) ─► split into chunks ─► embeddings (MiniLM)
                                                                     │
Question ─► embed ─► find top matching chunks ─► relevance check ──┬─► too low? → "Not related to the PDF"
                                                                   └─► ok → local LLM answers using ONLY those chunks (+ page numbers)
```

## Two layers that keep answers inside the PDF
1. **Relevance gate:** if the question does not match any part of the PDF (similarity below a threshold), the system refuses without calling the LLM.
2. **Strict prompt:** the LLM must answer only from the retrieved text, otherwise it replies `NOT_FOUND`, which is converted into a refusal.

> Note: works with text-based PDFs. Scanned (image-only) PDFs have no text to extract and need OCR.

## Step 1 — Install libraries

In [1]:
!pip install -q pypdf sentence-transformers accelerate
print("✅ Installed")

✅ Installed


## Step 2 — Imports and device

In [2]:
import re, time, warnings
import numpy as np
import torch
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

warnings.filterwarnings("ignore")
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device == "cpu":
    print("Tip: Runtime > Change runtime type > T4 GPU for faster answers.")

Device: cpu
Tip: Runtime > Change runtime type > T4 GPU for faster answers.


## Step 3 — Load local models (embedding model + LLM)

In [3]:
# Embedding model (for semantic search)
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)

# Local LLM (open-source, no API key)
LLM_NAME = "Qwen/Qwen2.5-1.5B-Instruct" if device == "cuda" else "Qwen/Qwen2.5-0.5B-Instruct"
dtype = torch.float16 if device == "cuda" else torch.float32

def load_llm(name):
    tok = AutoTokenizer.from_pretrained(name)
    mdl = AutoModelForCausalLM.from_pretrained(name, torch_dtype=dtype).to(device)
    mdl.eval()
    return tok, mdl

try:
    tokenizer, model = load_llm(LLM_NAME)
except Exception as e:
    print("Primary model failed, using fallback:", e)
    LLM_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
    tokenizer, model = load_llm(LLM_NAME)

print("✅ Models loaded. LLM:", LLM_NAME)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (14 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

✅ Models loaded. LLM: Qwen/Qwen2.5-0.5B-Instruct


In [4]:
def llm_generate(system_prompt, user_prompt, max_new_tokens=300):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False,
            temperature=None, top_p=None, top_k=None,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

print(llm_generate("You are a helpful assistant.", "Say hello in one short sentence.", 30))

Hello! It's nice to meet you.


## Step 4 — Upload your PDF(s)
Run this cell and choose one or more PDF files. (To change the PDF later, just run this cell and the next ones again.)

In [5]:
pdf_paths = []
try:
    from google.colab import files
    uploaded = files.upload()
    pdf_paths = [name for name in uploaded.keys() if name.lower().endswith(".pdf")]
except ImportError:
    path = input("Enter the PDF file path: ").strip()
    pdf_paths = [path]

if not pdf_paths:
    raise ValueError("No PDF uploaded. Please run this cell again and select a .pdf file.")
print("PDF files:", pdf_paths)

Saving Docker_Interview_Questions (1).pdf to Docker_Interview_Questions (1).pdf
PDF files: ['Docker_Interview_Questions (1).pdf']


## Step 5 — Extract text from the PDF (page by page)

In [6]:
pages = []
for path in pdf_paths:
    reader = PdfReader(path)
    if reader.is_encrypted:
        try:
            reader.decrypt("")
        except Exception:
            raise ValueError(f"{path} is password protected. Please upload an unlocked PDF.")
    for i, page in enumerate(reader.pages):
        text = page.extract_text() or ""
        text = re.sub(r"\s+", " ", text).strip()
        if text:
            pages.append({"file": path, "page": i + 1, "text": text})

if not pages:
    raise ValueError("No text found. This looks like a scanned/image PDF, which needs OCR. Please upload a text-based PDF.")

total_words = sum(len(p["text"].split()) for p in pages)
print(f"✅ Extracted text from {len(pages)} page(s), about {total_words} words.")
print("\nPreview of first page:\n", pages[0]["text"][:500], "...")

✅ Extracted text from 5 page(s), about 1313 words.

Preview of first page:
 Docker Interview Questions 30 Most Asked Real Interview Questions Q1. What is Docker and why is it used? Answer: Docker is a containerization platform that packages an application along with its dependencies into lightweight, portable containers. It ensures the application behaves consistently across development, testing, and production environments while simplifying deployment and scaling. Q2. What is the difference between a container and a virtual machine? Answer: A virtual machine includes a ...


## Step 6 — Chunking + Embeddings (build the searchable index)

In [7]:
def chunk_page(text, chunk_words=180, overlap=40):
    words = text.split()
    out, start = [], 0
    while start < len(words):
        out.append(" ".join(words[start:start + chunk_words]))
        if start + chunk_words >= len(words):
            break
        start += chunk_words - overlap
    return out

chunks = []
for p in pages:
    for c in chunk_page(p["text"]):
        chunks.append({"file": p["file"], "page": p["page"], "text": c})

chunk_embeddings = embedder.encode(
    [c["text"] for c in chunks], normalize_embeddings=True, show_progress_bar=False, batch_size=32
)
print("Total chunks:", len(chunks), "| Embedding shape:", chunk_embeddings.shape)

Total chunks: 10 | Embedding shape: (10, 384)


## Step 7 — Retrieval with a relevance gate
`RELEVANCE_THRESHOLD` controls how strict the system is. **Higher = stricter** (refuses more), **lower = more lenient**. Default 0.25 works well for most documents.

In [8]:
RELEVANCE_THRESHOLD = 0.25
TOP_K = 4
REFUSAL = "❌ Sorry, this question is not related to the uploaded PDF, so I can't answer it. Please ask something about the document."

def retrieve(question, k=TOP_K):
    q_emb = embedder.encode([question], normalize_embeddings=True, show_progress_bar=False)[0]
    scores = chunk_embeddings @ q_emb
    idx = np.argsort(-scores)[:k]
    return [{**chunks[i], "score": float(scores[i])} for i in idx]

def is_summary_question(q):
    q = q.lower()
    keys = ["summar", "overview", "what is this pdf", "what is this document", "what is this file",
            "about this pdf", "about this document", "main topic", "main points", "key points", "gist"]
    return any(k in q for k in keys)

def spread_chunks(n=6):
    """Pick chunks evenly across the whole document (for summary questions)."""
    if len(chunks) <= n:
        return [{**c, "score": 1.0} for c in chunks]
    idx = np.linspace(0, len(chunks) - 1, n).astype(int)
    return [{**chunks[i], "score": 1.0} for i in idx]

## Step 8 — Answer generation (strictly from the PDF)

In [9]:
SYSTEM_PROMPT = (
    "You are a document question-answering assistant. "
    "Answer the question using ONLY the provided PDF context. "
    "Do NOT use outside knowledge. "
    "If the answer is not clearly present in the context, reply with exactly: NOT_FOUND. "
    "Keep the answer clear and concise."
)
SUMMARY_PROMPT = (
    "You are a document assistant. Using ONLY the provided PDF excerpts, write a short, clear summary "
    "of what the document is about in 5 to 7 sentences. Do not add outside information."
)

def ask_pdf(question, show_sources=True, debug=False):
    start = time.time()
    question = question.strip()
    if not question:
        return None

    # Summary questions use chunks spread across the document
    if is_summary_question(question):
        docs = spread_chunks()
        context = "\n\n".join(f"[Page {d['page']}] {d['text']}" for d in docs)
        answer = llm_generate(SUMMARY_PROMPT, f"PDF excerpts:\n{context}\n\nWrite the summary:", 300)
        pages_used = sorted({d["page"] for d in docs})
    else:
        docs = retrieve(question)
        best = docs[0]["score"]
        if debug:
            print("Top similarity scores:", [round(d["score"], 3) for d in docs])
        # Layer 1: relevance gate
        if best < RELEVANCE_THRESHOLD:
            print(f"\n🙋 Question: {question}\n{REFUSAL}   (best match score {best:.2f} < {RELEVANCE_THRESHOLD})")
            return REFUSAL
        docs = [d for d in docs if d["score"] >= RELEVANCE_THRESHOLD * 0.8]
        context = "\n\n".join(f"[Page {d['page']}] {d['text']}" for d in docs)
        user_prompt = f"PDF context:\n{context}\n\nQuestion: {question}\n\nAnswer:"
        answer = llm_generate(SYSTEM_PROMPT, user_prompt, 300)
        # Layer 2: LLM said it is not in the PDF
        if "NOT_FOUND" in answer.upper().replace(" ", "_") or answer.strip() == "":
            print(f"\n🙋 Question: {question}\n{REFUSAL}   (the PDF does not contain this information)")
            return REFUSAL
        pages_used = sorted({d["page"] for d in docs})

    print(f"\n🙋 Question: {question}")
    print(f"🤖 Answer: {answer}")
    if show_sources:
        print(f"📖 Source pages: {pages_used}")
    print(f"⏱️ {time.time() - start:.1f}s")
    return answer

## Step 9 — Try it: summary and sample questions
Edit these questions to match your PDF.

In [10]:
_ = ask_pdf("Summarize this PDF")


🙋 Question: Summarize this PDF
🤖 Answer: This document provides a comprehensive overview of the key features and functionalities of Docker, including its core components (containers, images, and volumes), common usage scenarios, and best practices for deploying and managing Docker-based applications. It covers various aspects such as setting up Docker environments, building and running Docker images, managing containerized applications, and integrating Docker with CI/CD pipelines. The document also discusses Docker's role in container orchestration and its integration with popular platforms like Kubernetes.
📖 Source pages: [1, 2, 3, 4, 5]
⏱️ 38.4s


In [11]:
# 👉 Replace with a question related to YOUR pdf
_ = ask_pdf("What is themain topic discussed i this pdf?", debug=True)


🙋 Question: What is the main topic discussed in this document?
🤖 Answer: This document provides a comprehensive overview of the key features and functionalities of Docker, including its core components (containers, images, and volumes), common usage scenarios, and best practices for deploying and managing Docker-based applications. It covers various aspects such as setting up Docker environments, building and running Docker images, managing containerized applications, and integrating Docker with CI/CD pipelines. The document also discusses Docker's role in container orchestration and its integration with popular platforms like Kubernetes.
📖 Source pages: [1, 2, 3, 4, 5]
⏱️ 41.7s


## Step 10 — Test: unrelated questions must be refused
These questions have nothing to do with your PDF (unless your PDF is about them), so the system should refuse.

In [12]:
for q in ["Who won the cricket world cup?", "Write a poem about the moon", "What is the capital of France?"]:
    _ = ask_pdf(q, debug=True)

Top similarity scores: [-0.121, -0.122, -0.124, -0.125]

🙋 Question: Who won the cricket world cup?
❌ Sorry, this question is not related to the uploaded PDF, so I can't answer it. Please ask something about the document.   (best match score -0.12 < 0.25)
Top similarity scores: [-0.0, -0.029, -0.038, -0.041]

🙋 Question: Write a poem about the moon
❌ Sorry, this question is not related to the uploaded PDF, so I can't answer it. Please ask something about the document.   (best match score -0.00 < 0.25)
Top similarity scores: [0.043, 0.027, 0.027, 0.026]

🙋 Question: What is the capital of France?
❌ Sorry, this question is not related to the uploaded PDF, so I can't answer it. Please ask something about the document.   (best match score 0.04 < 0.25)


## Step 11 — Chat with your PDF (interactive)
Type your questions one by one. Type `exit` to stop.

In [13]:
print("💬 Chat with your PDF. Type 'exit' to stop.\n")
while True:
    q = input("Your question: ").strip()
    if q.lower() in ["exit", "quit", "q"]:
        print("Goodbye! 👋")
        break
    ask_pdf(q)

💬 Chat with your PDF. Type 'exit' to stop.

Your question: what is docker

🙋 Question: what is docker
🤖 Answer: Docker is a containerization platform that packages an application along with its dependencies into lightweight, portable containers. It ensures the application behaves consistently across development, testing, and production environments while simplifying deployment and scaling.
📖 Source pages: [1, 3, 4]
⏱️ 36.6s
Your question: what is image

🙋 Question: what is image
🤖 Answer: An image is a read-only template containing the application, runtime, libraries, dependencies, and configuration required to create containers.
📖 Source pages: [1, 2]
⏱️ 21.4s
Your question: who is the president of india

🙋 Question: who is the president of india
❌ Sorry, this question is not related to the uploaded PDF, so I can't answer it. Please ask something about the document.   (best match score -0.07 < 0.25)
Your question: exit
Goodbye! 👋


## 🔧 Tuning tips
| Problem | Fix |
|---|---|
| Relevant questions are refused | Lower `RELEVANCE_THRESHOLD` (e.g. 0.18) in Step 7 |
| Unrelated questions get answered | Raise `RELEVANCE_THRESHOLD` (e.g. 0.32) |
| Answers are incomplete | Increase `TOP_K` (e.g. 6) |
| Slow answers | Use a GPU runtime |
| "No text found" error | PDF is scanned images; it needs OCR |

Use `debug=True` in `ask_pdf(...)` to see the similarity scores while tuning.

## ✅ Conclusion
The system extracts text from any uploaded PDF, indexes it with embeddings, retrieves the most relevant passages for each question, and uses a local LLM to answer strictly from them, citing page numbers. Out-of-scope questions are rejected by the relevance gate and the strict prompt.